In [1]:
from mpramnist.Klein2020.dataset import KleinDataset
from mpramnist.Klein2020.trainer import LitModel_Klein

from mpramnist.Agarwal2025.dataset import AgarwalSingleDataset

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

from mpramnist import transforms as t

import torch
import torch.nn as nn
import lightning.pytorch as L

from torch.utils.data import DataLoader

from torchmetrics import PearsonCorrCoef
from lightning.pytorch.callbacks import ModelCheckpoint

In [2]:
train_transform = t.Compose(
    [
        t.AddFlanks(AgarwalSingleDataset.CONSTANT_LEFT_FLANK, AgarwalSingleDataset.CONSTANT_RIGHT_FLANK),
        t.AddFlanks("", AgarwalSingleDataset.RIGHT_FLANK),
        t.RightCrop(230, 260),  
        t.LeftCrop(230, 230),
        t.ReverseComplement(0.5),
        t.Seq2Tensor(),
    ]
)
test_transform = t.Compose(
    [
        t.AddFlanks(AgarwalSingleDataset.CONSTANT_LEFT_FLANK, AgarwalSingleDataset.CONSTANT_RIGHT_FLANK),
        t.ReverseComplement(0),
        t.Seq2Tensor(),
    ]
)

Cell_Type = "HepG2"
train_dataset = AgarwalSingleDataset(cell_type=Cell_Type, split="train", transform=train_transform, root="../data/",)
val_dataset = AgarwalSingleDataset(cell_type=Cell_Type, split="val", transform=test_transform, root="../data/",)
test_dataset = AgarwalSingleDataset(cell_type=Cell_Type, split="test", transform=test_transform, root="../data/",)
# encapsulate data into dataloader form
train_loader = DataLoader(dataset=train_dataset, batch_size=1024, shuffle=True, num_workers=103)
val_loader = DataLoader(dataset=val_dataset, batch_size=1024, shuffle=False, num_workers=103)
test_loader = DataLoader(dataset=test_dataset, batch_size=1024, shuffle=False, num_workers=103)

model = HumanLegNet(
    in_ch=4,
    output_dim=1,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model_HepG2 = LitModel_Klein(
    model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=5
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_pearson", mode="max", save_top_k=1, save_last=False
)

trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=5,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    callbacks=[checkpoint_callback],
)

trainer.fit(seq_model_HepG2, train_dataloaders=train_loader, val_dataloaders=val_loader)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.

Epoch 4: 100%|██████████| 97/97 [00:14<00:00,  6.53it/s, v_num=2, val_loss=0.320, val_pearson=0.685, train_loss=0.306]
-------------------------------------------------------------------------------
| Epoch: 4 | Val Loss: 0.30393 | Val Pearson: 0.70452 | Train Pearson: 0.74955 
-------------------------------------------------------------------------------

Epoch 4: 100%|██████████| 97/97 [00:20<00:00,  4.67it/s, v_num=2, val_loss=0.304, val_pearson=0.705, train_loss=0.268]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 97/97 [00:20<00:00,  4.65it/s, v_num=2, val_loss=0.304, val_pearson=0.705, train_loss=0.268]


In [3]:
best_model_path = checkpoint_callback.best_model_path
seq_model_hepg2 = LitModel_Klein.load_from_checkpoint(best_model_path, model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=1)
trainer.test(seq_model_hepg2, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 35.01it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.28957781195640564
      test_pearson           0.723478376865387
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.28957781195640564, 'test_pearson': 0.723478376865387}]

In [4]:
transform = t.Compose([t.Padding(230), t.Seq2Tensor(),])
rev_transform = t.Compose([t.Padding(230), t.ReverseComplement(1), t.Seq2Tensor(),])

exp = '5/3_WT'
# forward and reverse datasets
dataset = KleinDataset(experiment=exp,transform=transform, root = "../data")
rev_dataset = KleinDataset(experiment=exp,transform=rev_transform, root = "../data")
# forward and reverse datasets encapsulated into dataloaders
loader = DataLoader(dataset=dataset, batch_size=1024, shuffle=False, num_workers=103)
rev_loader = DataLoader(dataset=rev_dataset, batch_size=1024, shuffle=False, num_workers=103)

predictions_forw = trainer.predict(seq_model_hepg2, dataloaders=loader)
targets = torch.cat([pred["target"] for pred in predictions_forw])
y_preds_forw = torch.cat([pred["predicted"] for pred in predictions_forw])

predictions_rev = trainer.predict(seq_model_hepg2, dataloaders=rev_loader)
y_preds_rev = torch.cat([pred["predicted"] for pred in predictions_rev])

mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0).unsqueeze(-1)

pears = PearsonCorrCoef()
print(exp, " Pearson correlation")

print(pears(mean_forw, targets))

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 34.64it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 56.77it/s]
5/3_WT  Pearson correlation
tensor(0.7144)
